# 🎬 IMDB Sentiment Analysis — DistilBERT

---
## 📋 خطوات النوتبوك
1. تحميل الداتا من Kaggle
2. EDA سريعة
3. Tokenization + Dataset Class
4. Fine-tune DistilBERT
5. تقييم النتائج
6. Inference — تجربة Reviews جديدة
7. Batch Prediction من ملف CSV

## ⚙️ 0. Install Dependencies

In [ ]:
# شغّل مرة واحدة بس
# !pip install transformers torch scikit-learn pandas matplotlib seaborn kagglehub

## 📦 1. Imports & Config

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)
import kagglehub
import warnings
warnings.filterwarnings('ignore')

# ─── CONFIG ───────────────────────────────────────────────
MODEL_NAME  = 'distilbert-base-uncased'   # بدّله لـ roberta-base لنتيجة أحسن
MAX_LEN     = 256
BATCH_SIZE  = 32
EPOCHS      = 3
LR          = 2e-5
SEED        = 42
SAVE_PATH   = '../models/best_model.pt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch.manual_seed(SEED)
np.random.seed(SEED)

## 📊 2. Load & Explore Data

In [ ]:
path = kagglehub.dataset_download('lakshmi25npathi/imdb-dataset-of-50k-movie-reviews')
df   = pd.read_csv(os.path.join(path, 'IMDB Dataset.csv'))
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
df['sentiment'].value_counts().plot(
    kind='bar', ax=axes[0], color=['#ef4444', '#22c55e'],
    edgecolor='black', title='Class Distribution'
)
axes[0].set_xlabel('')
axes[0].tick_params(rotation=0)

# Review length distribution
df['review_len'] = df['review'].str.split().str.len()
axes[1].hist(df['review_len'], bins=50, color='#6366f1', edgecolor='black')
axes[1].axvline(df['review_len'].median(), color='red', linestyle='--', label=f'Median: {df["review_len"].median():.0f}')
axes[1].set_title('Review Length Distribution (words)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(df['review_len'].describe().round(1))

## ✂️ 3. Split Data

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['review'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=SEED,
    stratify=df['label']    # ✅ balanced
)

print(f'Train: {len(train_texts):,}  |  Test: {len(test_texts):,}')

## 🔤 4. Tokenizer & Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = IMDBDataset(train_texts, train_labels, tokenizer, MAX_LEN)
test_dataset  = IMDBDataset(test_texts,  test_labels,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}')

## 🤖 5. Model, Optimizer & Scheduler

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')
print(f'Warmup steps: {warmup_steps}  |  Total steps: {total_steps}')

## 🏋️ 6. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, preds_all, labels_all = 0, [], []
    for batch in loader:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        lbls  = batch['label'].to(device)
        optimizer.zero_grad()
        out   = model(input_ids=ids, attention_mask=mask, labels=lbls)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += out.loss.item()
        preds_all.extend(torch.argmax(out.logits, 1).cpu().numpy())
        labels_all.extend(lbls.cpu().numpy())
    return total_loss / len(loader), accuracy_score(labels_all, preds_all)

def evaluate(model, loader, device):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['label'].to(device)
            out  = model(input_ids=ids, attention_mask=mask)
            preds_all.extend(torch.argmax(out.logits, 1).cpu().numpy())
            labels_all.extend(lbls.cpu().numpy())
    return accuracy_score(labels_all, preds_all), f1_score(labels_all, preds_all), preds_all, labels_all

# ─── TRAIN ────────────────────────────────────────────────
history  = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}
best_acc = 0
os.makedirs('../models', exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    print(f'\n📌 Epoch {epoch}/{EPOCHS}')
    tr_loss, tr_acc    = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_acc, val_f1, _, _ = evaluate(model, test_loader, device)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    print(f'  Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.4f}')
    print(f'  Val Acc  : {val_acc:.4f} | Val F1  : {val_f1:.4f}')

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'  💾 Best model saved! Acc = {best_acc:.4f}')

print(f'\n🏆 Best Accuracy: {best_acc:.4f}')

## 📈 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, history['train_loss'], 'o-', color='#ef4444', label='Train Loss')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], 'o-', color='#6366f1', label='Train Acc')
axes[1].plot(epochs_range, history['val_acc'],   's-', color='#22c55e', label='Val Acc')
axes[1].plot(epochs_range, history['val_f1'],    '^-', color='#f59e0b', label='Val F1')
axes[1].set_title('Accuracy & F1'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 📊 8. Final Evaluation & Confusion Matrix

In [ ]:
model.load_state_dict(torch.load(SAVE_PATH))
final_acc, final_f1, y_pred, y_true = evaluate(model, test_loader, device)

print(f'📊 Test Accuracy : {final_acc:.4f}')
print(f'📊 Test F1 Score : {final_f1:.4f}')
print('\n', classification_report(y_true, y_pred, target_names=['Negative', 'Positive']))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 🎯 9. Single Review Inference

In [ ]:
def predict_sentiment(text, model, tokenizer, device, max_len=256):
    model.eval()
    enc = tokenizer(
        text, max_length=max_len,
        padding='max_length', truncation=True,
        return_tensors='pt'
    )
    with torch.no_grad():
        out   = model(
            input_ids=enc['input_ids'].to(device),
            attention_mask=enc['attention_mask'].to(device)
        )
    proba  = torch.softmax(out.logits, dim=1)[0].cpu().numpy()
    pred   = int(np.argmax(proba))
    label  = 'Positive 😊' if pred == 1 else 'Negative 😞'
    return {
        'label':      label,
        'prediction': pred,
        'confidence': float(proba[pred]),
        'prob_neg':   float(proba[0]),
        'prob_pos':   float(proba[1])
    }

# ─── جرّب هنا ────────────────────────────────────────────
test_reviews = [
    'This movie was absolutely fantastic! The acting was superb.',
    'Total waste of time. Boring plot and terrible acting.',
    'It was okay, nothing special but not bad either.',
]

for review in test_reviews:
    result = predict_sentiment(review, model, tokenizer, device)
    print(f'Review   : {review[:60]}...')
    print(f'Result   : {result["label"]}  (confidence: {result["confidence"]:.2%})')
    print(f'Probs    : Neg={result["prob_neg"]:.3f}  Pos={result["prob_pos"]:.3f}')
    print('-' * 60)

## 📂 10. Batch Prediction from CSV

In [ ]:
def predict_batch_csv(input_csv, output_csv, model, tokenizer, device,
                       text_col='review', batch_size=64, max_len=256):
    """
    input_csv  : ملف فيه عمود review (أو أي اسم تحدده في text_col)
    output_csv : هيتحفظ فيه مع إضافة عمودين: predicted_label و confidence
    """
    df_in = pd.read_csv(input_csv)
    texts = df_in[text_col].tolist()

    all_labels, all_conf = [], []
    model.eval()

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i: i + batch_size]
        enc = tokenizer(
            batch_texts, max_length=max_len,
            padding=True, truncation=True,
            return_tensors='pt'
        )
        with torch.no_grad():
            out   = model(
                input_ids=enc['input_ids'].to(device),
                attention_mask=enc['attention_mask'].to(device)
            )
        proba = torch.softmax(out.logits, dim=1).cpu().numpy()
        preds = np.argmax(proba, axis=1)
        all_labels.extend(['positive' if p == 1 else 'negative' for p in preds])
        all_conf.extend([float(proba[j][preds[j]]) for j in range(len(preds))])

        if (i // batch_size + 1) % 10 == 0:
            print(f'  Processed {min(i+batch_size, len(texts)):,}/{len(texts):,}')

    df_in['predicted_label'] = all_labels
    df_in['confidence']      = [f'{c:.4f}' for c in all_conf]
    df_in.to_csv(output_csv, index=False)
    print(f'\n✅ Saved to: {output_csv}  ({len(df_in):,} rows)')
    return df_in

# ─── مثال استخدام ────────────────────────────────────────
# results = predict_batch_csv(
#     input_csv  = '../data/sample/sample_reviews.csv',
#     output_csv = '../data/sample/predictions.csv',
#     model      = model,
#     tokenizer  = tokenizer,
#     device     = device
# )
# results.head()

print('✅ predict_batch_csv جاهزة — شيل التعليق فوق وجرّبها!')